In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
import time
import seaborn as sns
import matplotlib.pyplot as plt


# Load data
df = pd.read_csv('/kaggle/input/telecom-customer/Telecom_customer churn.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/telecom-customer/Telecom_customer churn.csv'

Let's show the statistical summary of both categorical and numerical columns, respectively.

In [ ]:
# Only Categorical columns
df.describe(include=['O'])

In [ ]:
# Only numerical columns
df.describe()

In [ ]:
df.columns

In [ ]:
# Count missing values for each column
for col in df.columns:
    if(df[col].isnull().sum()>0):
        print(col, '->', df[col].isnull().sum())

## Exploratory Data Analysis (EDA)

In [ ]:
cat_cols = df.select_dtypes(include='object').columns
cat_cols

In [ ]:
for col in cat_cols:
    sns.countplot(x=col, hue="churn", data=df)
    plt.xticks()
    plt.show()

In [ ]:
# ratio of those who churn and those who didn't
sizes = [48401, 47647]
labels = 'NO', 'YES'
explode = (0, 0.1)
fig, ax = plt.subplots()
ax.pie(sizes, explode=explode, autopct='%1.1f%%', shadow=True, startangle=75)
ax.axis('equal')
ax.set_title("Client Churn Distribution")
ax.legend(labels)
plt.show()

Finding missing values and filling appropriately

In [ ]:
df.columns[df.isnull().any()]

In [ ]:
# Let's drop the columns that seem to have no significant contribution to the model.
df.drop(['numbcars', 'dwllsize', 'HHstatin', 'ownrent',
         'dwlltype','lor','income','adults','prizm_social_one',
         'infobase','crclscod'], axis=1, inplace=True)

In [ ]:
df['hnd_webcap']=df['hnd_webcap'].fillna('UNKW') # Handset web capability

df['avg6qty']=df['avg6qty'].fillna(df['avg6qty'].mean()) # Billing adjusted total number of calls over the life of the customer
df['avg6rev']=df['avg6rev'].fillna(df['avg6rev'].mean()) # Average monthly revenue over the life of the customer
df['avg6mou']=df['avg6mou'].fillna(df['avg6mou'].mean()) # Average monthly minutes of use over the life of the customer

df['change_mou']=df['change_mou'].fillna(df['change_mou'].mean()) # Percentage change in monthly minutes of use vs previous three month average
df['change_rev']=df['change_rev'].fillna(df['change_rev'].mean()) # Percentage change in monthly revenue vs previous three month average

df['rev_Mean']=df['rev_Mean'].fillna(df['rev_Mean'].mean())
df['totmrc_Mean']=df['totmrc_Mean'].fillna(df['totmrc_Mean'].mean())
df['da_Mean']=df['da_Mean'].fillna(df['da_Mean'].mean())
df['ovrmou_Mean']=df['ovrmou_Mean'].fillna(df['ovrmou_Mean'].mean())
df['ovrrev_Mean']=df['ovrrev_Mean'].fillna(df['ovrrev_Mean'].mean())
df['vceovr_Mean']=df['vceovr_Mean'].fillna(df['vceovr_Mean'].mean())
df['datovr_Mean']=df['datovr_Mean'].fillna(df['datovr_Mean'].mean())
df['roam_Mean']=df['roam_Mean'].fillna(df['roam_Mean'].mean())
df['mou_Mean']=df['mou_Mean'].fillna(df['mou_Mean'].mean())

df.dropna(inplace=True)

## Non-Distributed Implementation

In [ ]:
y = df['churn']  # Keep this as the target
X = df.drop(labels=['Customer_ID', 'churn'], axis=1)  # Features without 'churn' and also Customer_ID is a unique identifier and not significant for model

# Identify categorical features (EXCLUDING 'churn')
categorical_features = X.select_dtypes(include='object').columns.tolist()
numerical_features = X.select_dtypes(exclude='object').columns.tolist()

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(), categorical_features),
        ('num', 'passthrough', numerical_features)
    ])

# Transform features (X)
X_processed = preprocessor.fit_transform(X)

# Encode target (y)
y_encoded = LabelEncoder().fit_transform(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_processed, y_encoded, test_size=0.3, random_state=42)

In [ ]:
# Train model
start_time = time.time()
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
non_distributed_time = time.time() - start_time

In [ ]:
# Evaluate
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Non-Distributed Accuracy: {accuracy:.4f}, Time: {non_distributed_time:.2f}s")

## Distributed Implementation

In [ ]:
from pyspark.ml.classification import RandomForestClassifier as SparkRFC
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, Imputer
from pyspark.ml import Pipeline
from pyspark.sql.types import *

# Initialize Spark session
spark = SparkSession.builder \
    .appName("ChurnPrediction") \
    .master("local[*]") \
    .getOrCreate()

# Load data
df_spark = spark.read.csv('/kaggle/input/telecom-customer/Telecom_customer churn.csv', header=True, inferSchema=True)

# Drop columns (same as pandas version)
columns_to_drop = ['numbcars', 'dwllsize', 'HHstatin', 'ownrent', 'dwlltype', 'lor',
                  'income', 'adults', 'prizm_social_one', 'infobase', 'crclscod',
                  'Customer_ID']
df_spark = df_spark.drop(*columns_to_drop)

# Handle missing values
# For categorical column
df_spark = df_spark.fillna({'hnd_webcap': 'UNKW'})

# For numerical columns - calculate mean and fill
numerical_columns = ['avg6qty', 'avg6rev', 'avg6mou', 'change_mou', 'change_rev',
                    'rev_Mean', 'totmrc_Mean', 'da_Mean', 'ovrmou_Mean', 'ovrrev_Mean',
                    'vceovr_Mean', 'datovr_Mean', 'roam_Mean', 'mou_Mean']

# Create imputers for numerical columns
imputers = [Imputer(inputCols=[col], outputCols=[col], strategy='mean')
           for col in numerical_columns]

# Create pipeline for imputation
imputation_pipeline = Pipeline(stages=imputers)
df_spark = imputation_pipeline.fit(df_spark).transform(df_spark)

# Drop any remaining null values
df_spark = df_spark.na.drop()

# Separate features and target
y = df_spark.select('churn')
X = df_spark.drop('churn')

In [ ]:
full_data = df_spark  # the complete dataset

# Identify categorical/numerical columns
categorical_columns = [f.name for f in full_data.schema.fields
                      if isinstance(f.dataType, StringType) and f.name != 'churn']
numerical_columns = [f.name for f in full_data.schema.fields
                   if isinstance(f.dataType, (IntegerType, DoubleType, FloatType)) and f.name != 'churn']

# Modified preprocessing pipeline
indexers = [StringIndexer(inputCol=col, outputCol=f"{col}_index", handleInvalid="keep")
          for col in categorical_columns]

encoder = OneHotEncoder(
    inputCols=[f"{col}_index" for col in categorical_columns],
    outputCols=[f"{col}_encoded" for col in categorical_columns]
)

assembler = VectorAssembler(
    inputCols=[f"{col}_encoded" for col in categorical_columns] + numerical_columns,
    outputCol="features"
)

# Add StringIndexer for the target column 'churn'
label_indexer = StringIndexer(inputCol="churn", outputCol="label")

In [ ]:
preprocessing_pipeline = Pipeline(stages=indexers + [encoder, assembler, label_indexer])

# Transform FULL dataset (including 'churn')
preprocessed_data = preprocessing_pipeline.fit(full_data).transform(full_data)

# Now select features and label
final_data = preprocessed_data.select("features", "label")

# Train-test split
train_data, test_data = final_data.randomSplit([0.7, 0.3], seed=42)

In [ ]:
# Train model
start_time = time.time()
spark_clf = SparkRFC(labelCol="label", numTrees=100, seed=42)
spark_model = spark_clf.fit(train_data)
distributed_time = time.time() - start_time

In [ ]:
# Evaluate
predictions = spark_model.transform(test_data)
accuracy = predictions.filter(predictions.label == predictions.prediction).count() / test_data.count()
print(f"Distributed Accuracy: {accuracy:.4f}, Time: {distributed_time:.2f}s")

spark.stop()

## Measure Speed-Up, Size-Up, and Scale-Up metrics

In [ ]:
########################################################################### Measure Speed-Up (Vary cores, fixed data size) #############################################################

speed_up_cores = [1, 2, 4]  # Test with 1, 2, and 4 cores
speed_up_times = []

for cores in speed_up_cores:
    print(f"\n=== Speed-Up Test: {cores} core(s) ===")

    # Initialize Spark with core configuration
    spark = SparkSession.builder \
        .appName(f"SpeedUp_{cores}cores") \
        .master(f"local[{cores}]") \
        .getOrCreate()

    # Load and preprocess data (same as original distributed code)
    df_spark = spark.read.csv('/kaggle/input/telecom-customer/Telecom_customer churn.csv',
                             header=True, inferSchema=True)

    # --- Preprocessing Steps (Same as Original) ---
    df_spark = df_spark.drop(*columns_to_drop)
    df_spark = df_spark.fillna({'hnd_webcap': 'UNKW'})
    imputation_pipeline = Pipeline(stages=imputers)
    df_spark = imputation_pipeline.fit(df_spark).transform(df_spark)
    df_spark = df_spark.na.drop()
    preprocessed_data = preprocessing_pipeline.fit(df_spark).transform(df_spark)
    final_data = preprocessed_data.select("features", "label")
    train_data, _ = final_data.randomSplit([0.7, 0.3], seed=42)
    # ----------------------------------------------

    # Time training
    start_time = time.time()
    spark_clf = SparkRFC(labelCol="label", numTrees=100, seed=42)
    spark_model = spark_clf.fit(train_data)
    speed_up_times.append(time.time() - start_time)

    spark.stop()
    print(f"Time with {cores} core(s): {speed_up_times[-1]:.2f}s")

# Calculate Speed-Up ratios
base_time = speed_up_times[0]
speed_up_ratios = [base_time / t for t in speed_up_times]

# Plot
plt.figure(figsize=(10,6))
plt.plot(speed_up_cores, speed_up_ratios, 'bo-', label='Actual Speed-Up')
plt.plot(speed_up_cores, speed_up_cores, 'r--', label='Ideal Linear Speed-Up')
plt.xlabel('Number of Cores'); plt.ylabel('Speed-Up (T1/Tn)')
plt.title('Speed-Up Measurement'); plt.legend(); plt.grid()
plt.show()








#############################################################################   Measure Size-Up (Fixed cores, varying data size)  ###############################################



size_up_multipliers = [1, 2, 4]  # Data size multipliers
size_up_times = []
FIXED_CORES = 4  # Keep cores constant

for multiplier in size_up_multipliers:
    print(f"\n=== Size-Up Test: {multiplier}x Data ===")

    spark = SparkSession.builder \
        .appName(f"SizeUp_{multiplier}x") \
        .master(f"local[{FIXED_CORES}]") \
        .getOrCreate()

    # Load and replicate data
    df_spark_orig = spark.read.csv('/kaggle/input/telecom-customer/Telecom_customer churn.csv',
                                  header=True, inferSchema=True)
    replicated_df = df_spark_orig
    for _ in range(multiplier - 1):
        replicated_df = replicated_df.union(df_spark_orig)

    # --- Preprocessing Steps ---
    replicated_df = replicated_df.drop(*columns_to_drop)
    replicated_df = replicated_df.fillna({'hnd_webcap': 'UNKW'})
    imputation_pipeline = Pipeline(stages=imputers)
    replicated_df = imputation_pipeline.fit(replicated_df).transform(replicated_df)
    replicated_df = replicated_df.na.drop()
    preprocessed_data = preprocessing_pipeline.fit(replicated_df).transform(replicated_df)
    final_data = preprocessed_data.select("features", "label")
    train_data, _ = final_data.randomSplit([0.7, 0.3], seed=42)
    # ---------------------------

    # Time training
    start_time = time.time()
    spark_clf = SparkRFC(labelCol="label", numTrees=100, seed=42)
    spark_model = spark_clf.fit(train_data)
    size_up_times.append(time.time() - start_time)

    spark.stop()
    print(f"Time with {multiplier}x data: {size_up_times[-1]:.2f}s")

# Calculate Size-Up ratios
base_size_time = size_up_times[0]
size_up_ratios = [t / base_size_time for t in size_up_times]

# Plot
plt.figure(figsize=(10,6))
plt.plot(size_up_multipliers, size_up_ratios, 'bo-', label='Actual Size-Up')
plt.plot(size_up_multipliers, size_up_multipliers, 'r--', label='Ideal Linear')
plt.xlabel('Data Size Multiplier'); plt.ylabel('Time Ratio (Tn/T1)')
plt.title('Size-Up Measurement'); plt.legend(); plt.grid()
plt.show()










############################################################     Measure Scale-Up (Increase data & cores proportionally) ##################################################

scale_up_multipliers = [1, 2]  # Multipliers for data and cores
scale_up_times = []

for scale in scale_up_multipliers:
    print(f"\n=== Scale-Up Test: {scale}x Data & Cores ===")

    spark = SparkSession.builder.appName(f"ScaleUp_{scale}x").master(f"local[{scale}]").getOrCreate()

    # Load and replicate data
    df_spark_orig = spark.read.csv('/kaggle/input/telecom-customer/Telecom_customer churn.csv',
                                  header=True, inferSchema=True)
    replicated_df = df_spark_orig
    for _ in range(scale - 1):
        replicated_df = replicated_df.union(df_spark_orig)

    # --- Preprocessing Steps ---
    replicated_df = replicated_df.drop(*columns_to_drop)
    replicated_df = replicated_df.fillna({'hnd_webcap': 'UNKW'})
    imputation_pipeline = Pipeline(stages=imputers)
    replicated_df = imputation_pipeline.fit(replicated_df).transform(replicated_df)
    replicated_df = replicated_df.na.drop()
    preprocessed_data = preprocessing_pipeline.fit(replicated_df).transform(replicated_df)
    final_data = preprocessed_data.select("features", "label")
    train_data, _ = final_data.randomSplit([0.7, 0.3], seed=42)
    # ---------------------------

    # Time training
    start_time = time.time()
    spark_clf = SparkRFC(labelCol="label", numTrees=100, seed=42)
    spark_model = spark_clf.fit(train_data)
    scale_up_times.append(time.time() - start_time)

    spark.stop()
    print(f"Time with {scale}x data/cores: {scale_up_times[-1]:.2f}s")

# Plot
plt.figure(figsize=(10,6))
plt.plot(scale_up_multipliers, scale_up_times, 'bo-', label='Actual Time')
plt.xlabel('Scale Multiplier (Data & Cores)'); plt.ylabel('Time (seconds)')
plt.title('Scale-Up Measurement'); plt.grid()
plt.show()

#